In [2]:
# ==== Environment ====

import os
import getpass
import numpy as np
import pandas as pd
import wrds
from sklearn.preprocessing import minmax_scale
from datetime import datetime
import gc
import warnings
warnings.filterwarnings('ignore')
import sys

In [3]:
# Create temp directory
(
    os
    .makedirs("temp", 
              exist_ok=True)
)

In [4]:
# ==== Helper Functions ====

# Convert date to month dummy
def md(caldt, fmt="%Y-%m-%d"):
    dt = datetime.strptime(str(caldt), fmt)
        
    return dt.year * 12 + dt.month - (1925 * 12 + 11)


# Rank and scale to [-1, +1]
def rank_scale(series):
    ranked = series.rank(method="average", na_option="keep")
    
    if ranked.notna().any():
        scaled = minmax_scale(ranked, feature_range=(-1, 1))
        return pd.Series(scaled, index=series.index)
        
    else:
        return series  # if all NaN

In [5]:
# ==== Login To WRDS ====
# Open a connection
db = wrds.Connection(wrds_username="nglei2025") # will ask for password if no .pgpass

Loading library list...
Done


In [8]:
# ==== Download and Process Monthly CRSP Stuff ====

query = """
SELECT a.permno, a.date, a.ret, a.shrout, a.prc,
       b.exchcd,
       c.dlstcd, c.dlret
FROM crsp.msf AS a
LEFT JOIN crsp.msenames AS b
  ON a.permno = b.permno
 AND b.namedt <= a.date
 AND a.date   <= b.nameendt
LEFT JOIN crsp.msedelist AS c
  ON a.permno = c.permno
 AND date_trunc('month', a.date) = date_trunc('month', c.dlstdt)
WHERE a.date >= '2010-01-01'
LIMIT 10;
"""
crspm = db.raw_sql(query)

# Handle delisting returns
num_cols = ["dlret", "ret", "shrout", "prc"]
crspm[num_cols] = crspm[num_cols].apply(pd.to_numeric, errors="coerce")
for col in ["dlstcd", "exchcd"]:
    crspm[col] = pd.to_numeric(crspm[col], errors="coerce").astype("Int64")

# --- 2) NA-safe masks for delisting imputations ---
dlret_na   = crspm["dlret"].isna()
dl_codes   = (crspm["dlstcd"].eq(500) | crspm["dlstcd"].between(520, 584, inclusive="both")).fillna(False)
nyse_amex  = crspm["exchcd"].isin([1, 2]).fillna(False)
nasdaq     = crspm["exchcd"].eq(3).fillna(False)

# --- 3) Impute dlret by exchange; cap; fill remaining ---
crspm.loc[dlret_na & dl_codes & nyse_amex, "dlret"] = -0.35
crspm.loc[dlret_na & dl_codes & nasdaq,    "dlret"] = -0.55
crspm["dlret"] = crspm["dlret"].clip(lower=-1).fillna(0)

# --- 4) Adjust returns with delisting; backfill NaN ret with dlret when dlret != 0 ---
crspm["ret"] = (1 + crspm["ret"]) * (1 + crspm["dlret"]) - 1
crspm.loc[crspm["ret"].isna() & (crspm["dlret"] != 0), "ret"] = crspm["dlret"]

# --- 5) Dates and derived fields ---
crspm["date"]   = pd.to_datetime(crspm["date"], errors="coerce")
crspm["me"]     = (crspm["prc"].abs() * crspm["shrout"]).replace([np.inf, -np.inf], np.nan)
crspm["yyyymm"] = crspm["date"].dt.year * 100 + crspm["date"].dt.month

# --- 6) Signals (avoid log of non-positive by nulling then logging) ---
# price: if prc <= 0 -> NaN, then log; same for me
price_pos = crspm["prc"].abs().where(crspm["prc"].abs() > 0)
me_pos    = crspm["me"].where(crspm["me"] > 0)

# NOTE THESE ARE SIGNED!
crspmsignal = pd.DataFrame({
    "permno":     crspm["permno"],
    "yyyymm":     crspm["yyyymm"],
    "STreversal": -crspm["ret"].fillna(0),
    "Price":      -np.log(price_pos),
    "Size":       -np.log(me_pos)
})

In [14]:
# ==== Read Wide Predictors ====
wide_dl_raw = pd.read_csv("temp/signed_predictors_dl_wide.csv")

KeyboardInterrupt: 

In [10]:
# ==== Merge And Export CSV ====
signalwide =\
(
    wide_dl_raw
    .merge(crspmsignal,
           on=["permno", "yyyymm"],
           how="outer")
)

wide_cz_data = signalwide.copy()

wide_cz_data["md"] =\
(
    (wide_cz_data["yyyymm"]
     .astype(str) + "15")
    .apply(lambda d: md(d, "%Y%m%d") + 1)
)

wide_cz_data["year"] = wide_cz_data["yyyymm"] // 100

wide_cz_data["month"] = wide_cz_data["yyyymm"] % 100

NameError: name 'wide_dl_raw' is not defined

In [9]:
# ==== Fama-French RF ====
ff3 = pd.read_csv("ff3.csv")
    
ff3["rf"] = ff3["rf"] / 100

ff3 = ff3[["yyyymm", "rf"]]

ret = crspm.merge(ff3, on="yyyymm", how="left")

ret["md"] =\
(
    (ret["yyyymm"]
     .astype(str) + "15")
    .apply(lambda d: md(d, "%Y%m%d"))
)

ret["mexret"] = ret["ret"] - ret["rf"]

ret = ret[["permno", "yyyymm", "md", "mexret"]].rename(columns={"yyyymm": "yyyymm_ret"})

final_cz_wide = wide_cz_data.merge(ret, on=["md", "permno"], how="left")

final_cz_wide.to_csv("temp/signed_predictors_all_wide.csv", index=False)

gc.collect()

NameError: name 'crspm' is not defined

In [ ]:
# ==== Clean and Scale ====
final_cz_wide_nona = final_cz_wide.dropna(subset=["mexret"])

SignalDoc = pd.read_csv("temp/SignalDoc.csv")


regs_char_cz =\
(
    SignalDoc
    .query("Cat.Signal=='Predictor'")["Acronym"]
    .tolist()
)

final_cz_wide_nona["counter"] = final_cz_wide_nona["md"] - final_cz_wide_nona["md"].min() + 1

def scale_group(df):
    for col in regs_char_cz:
        if col in df.columns:
            df[col] = rank_scale(df[col])
            df[col] = df[col].fillna(0)
    return df

final_cz_wide_final =\
(
    final_cz_wide_nona
    .groupby("md", group_keys=False)
    .apply(scale_group)
)

# Filter sample
final_cz_wide_final_3 = final_cz_wide_final.query("yyyymm >= 196307")

final_cz_wide_final_3.to_csv("temp/final_cz207_63.csv", index=False)
final_cz_wide_final_3.drop(columns=["STreversal"]).to_csv("temp/final_cz206_63.csv", index=False)

cz_counter_md = (final_cz_wide_final[["counter", "md", "yyyymm_ret", "year", "month"]]
                 .drop_duplicates()
                 .sort_values("counter"))
cz_counter_md.to_csv("temp/cz_counter_md.csv", index=False)

gc.collect()